# LLM Benchmark Notebook


It is designed so you can:
- update configuration in one place
- run connectivity checks before the full benchmark
- execute benchmarks cell by cell
- see tables and plots directly in notebook output
- still save CSV, JSON, PNG, and an HTML report for client sharing

The benchmark uses:
- `POST {BASE_URL}/v1/completions`
- Bearer token read from `/etc/secrets/ezua/.auth_token` on each request
- payload format:
  ```json
  {"model": "...", "prompt": "...", "max_tokens": ...}
  ```
- response format:
  ```json
  {"choices":[{"text":"..."}], "usage": {...}}
  ```


In [ ]:

# Install if needed
# !pip install httpx nest_asyncio pandas matplotlib tiktoken -q


In [ ]:

import asyncio
import json
import math
import os
import statistics
import time
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Optional, List, Dict, Any

import httpx
import nest_asyncio
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML

nest_asyncio.apply()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1. Configuration

Update these values before running the benchmark.


In [ ]:

# =========================
# CONFIGURATION
# =========================

# BASE_URL = "https://llama-3-1-8b-instruct-1-predictor-ashish-kumarj-h-653cd6af.ingress.pcai.hpelabs.co.il"
BASE_URL = os.getenv("BENCHMARK_BASE_URL", "https://your-inference-service-url")

MODEL_NAME = os.getenv("BENCHMARK_MODEL_NAME", "meta/llama-3.1-8b-instruct")

TOKEN_FILE_PATH = os.getenv("BENCHMARK_TOKEN_FILE", "/etc/secrets/ezua/.auth_token")

OUTPUT_DIR = Path(os.getenv("BENCHMARK_OUTPUT_DIR", "benchmark_output_notebook"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_LENGTHS = [512, 1024, 2048, 4096]
CONCURRENCY_LEVELS = [1, 2, 4, 8]

REQUESTS_PER_TEST = 20
WARMUP_REQUESTS = 3

MAX_OUTPUT_TOKENS = 128
TIMEOUT_SECONDS = 180.0
VERIFY_SSL = False
TEMPERATURE = 0.0

USER_TASK_PREFIX = "Summarize the following benchmark content accurately:\n\n"

USE_TOKENIZER = True
TOKENIZER_ENCODING = "cl100k_base"

print("BASE_URL:", BASE_URL)
print("MODEL_NAME:", MODEL_NAME)
print("TOKEN_FILE_PATH:", TOKEN_FILE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


## 2. Tokenizer setup

In [ ]:

tokenizer = None
if USE_TOKENIZER:
    try:
        import tiktoken
        tokenizer = tiktoken.get_encoding(TOKENIZER_ENCODING)
        print(f"[INFO] Using tokenizer encoding: {TOKENIZER_ENCODING}")
    except Exception as e:
        print(f"[WARN] tiktoken unavailable. Falling back to rough estimate. Error: {e}")
        tokenizer = None
        USE_TOKENIZER = False


## 3. Helper functions and data structures

In [ ]:

@dataclass
class RequestResult:
    context_length: int
    concurrency: int
    phase: str
    request_id: int
    success: bool
    status_code: Optional[int]
    total_latency_ms: Optional[float]
    error: Optional[str]
    response_chars: int
    prompt_tokens: Optional[int]
    completion_tokens: Optional[int]
    total_tokens: Optional[int]
    output_tokens_est: Optional[int]

def percentile(values: List[float], p: float) -> Optional[float]:
    if not values:
        return None
    if len(values) == 1:
        return float(values[0])
    values = sorted(values)
    k = (len(values) - 1) * (p / 100.0)
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return float(values[int(k)])
    return float(values[f] * (c - k) + values[c] * (k - f))

def count_tokens(text: str) -> int:
    if tokenizer is not None:
        return len(tokenizer.encode(text))
    return max(1, int(len(text.split()) * 1.3))

def build_prompt_exact_tokens(target_context_length: int) -> str:
    seed = USER_TASK_PREFIX
    filler = (
        "This is synthetic benchmark content used to simulate a long document page or context window. "
        "It includes repeated information about infrastructure, model inference, deployment benchmarking, "
        "latency measurement, concurrency validation, request handling, response generation, token usage, "
        "and controlled performance testing for enterprise model-serving environments. "
    )

    if tokenizer is None:
        text = seed
        while count_tokens(text) < target_context_length:
            text += filler
        return text

    seed_tokens = tokenizer.encode(seed)
    filler_tokens = tokenizer.encode(filler)

    if len(seed_tokens) >= target_context_length:
        return tokenizer.decode(seed_tokens[:target_context_length])

    current_tokens = list(seed_tokens)
    while len(current_tokens) + len(filler_tokens) <= target_context_length:
        current_tokens.extend(filler_tokens)

    remaining = target_context_length - len(current_tokens)
    if remaining > 0:
        current_tokens.extend(filler_tokens[:remaining])

    return tokenizer.decode(current_tokens)

def read_auth_token() -> str:
    with open(TOKEN_FILE_PATH, "r", encoding="utf-8") as f:
        token = f.read().strip()
    if not token:
        raise RuntimeError(f"Token file is empty: {TOKEN_FILE_PATH}")
    return token

def build_headers() -> Dict[str, str]:
    token = read_auth_token()
    return {
        "accept": "application/json",
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}",
    }

def build_payload(prompt: str, max_output_tokens: int) -> Dict[str, Any]:
    return {
        "model": MODEL_NAME,
        "prompt": prompt,
        "max_tokens": max_output_tokens,
        "temperature": TEMPERATURE,
    }

def format_request_error(exc: Exception) -> str:
    response = getattr(exc, "response", None)
    if response is None:
        return str(exc)
    body = None
    try:
        payload = response.json()
        if isinstance(payload, dict):
            body = payload.get("error") or payload.get("message") or payload.get("detail")
        if body is None:
            body = json.dumps(payload)[:500]
    except Exception:
        try:
            body = response.text[:500]
        except Exception:
            body = None
    if body:
        return f"{exc} | response: {body}"
    return str(exc)

def print_prompt_stats():
    rows = []
    for cl in CONTEXT_LENGTHS:
        prompt = build_prompt_exact_tokens(cl)
        rows.append({
            "target_context_length": cl,
            "actual_prompt_tokens": count_tokens(prompt),
            "preview": prompt[:120].replace("\n", " ") + "..."
        })
    df = pd.DataFrame(rows)
    display(df)
    return df


## 4. Prompt verification

This lets you verify that the generated prompts are close to the target token lengths before sending any benchmark traffic.


In [ ]:

prompt_verification_df = print_prompt_stats()


## 5. Quick token / auth sanity checks

In [ ]:

token = read_auth_token()
print("Token read successfully.")
print("Token preview:", token[:20] + "..." if len(token) > 20 else token)
print("Headers ready.")
build_headers()


## 6. Single request connectivity test

Run this before the full benchmark.


In [ ]:

async def single_request(
    client: httpx.AsyncClient,
    base_url: str,
    prompt: str,
    max_output_tokens: int,
    timeout_s: float,
    context_length: int,
    concurrency: int,
    request_id: int,
    phase: str,
) -> RequestResult:
    url = f"{base_url.rstrip('/')}/v1/completions"
    start = time.perf_counter()
    response_chars = 0

    try:
        headers = build_headers()
        payload = build_payload(prompt, max_output_tokens)

        response = await client.post(
            url,
            json=payload,
            headers=headers,
            timeout=timeout_s,
        )

        status_code = response.status_code
        response.raise_for_status()
        data = response.json()

        text = data.get("choices", [{}])[0].get("text", "")
        response_chars = len(text)
        usage = data.get("usage", {}) if isinstance(data, dict) else {}

        total_latency_ms = (time.perf_counter() - start) * 1000

        return RequestResult(
            context_length=context_length,
            concurrency=concurrency,
            phase=phase,
            request_id=request_id,
            success=True,
            status_code=status_code,
            total_latency_ms=round(total_latency_ms, 2),
            error=None,
            response_chars=response_chars,
            prompt_tokens=usage.get("prompt_tokens"),
            completion_tokens=usage.get("completion_tokens"),
            total_tokens=usage.get("total_tokens"),
            output_tokens_est=count_tokens(text) if text else 0,
        )

    except Exception as e:
        total_latency_ms = (time.perf_counter() - start) * 1000
        status_code = getattr(getattr(e, "response", None), "status_code", None)
        return RequestResult(
            context_length=context_length,
            concurrency=concurrency,
            phase=phase,
            request_id=request_id,
            success=False,
            status_code=status_code,
            total_latency_ms=round(total_latency_ms, 2),
            error=format_request_error(e),
            response_chars=response_chars,
            prompt_tokens=None,
            completion_tokens=None,
            total_tokens=None,
            output_tokens_est=None,
        )

async def connectivity_test():
    transport = httpx.AsyncHTTPTransport(retries=0, verify=VERIFY_SSL)
    async with httpx.AsyncClient(transport=transport) as client:
        prompt = "Once upon a time"
        result = await single_request(
            client=client,
            base_url=BASE_URL,
            prompt=prompt,
            max_output_tokens=32,
            timeout_s=TIMEOUT_SECONDS,
            context_length=len(prompt.split()),
            concurrency=1,
            request_id=1,
            phase="connectivity_test",
        )
        return result

connectivity_result = asyncio.run(connectivity_test())
display(pd.DataFrame([asdict(connectivity_result)]))


## 7. Batch execution helpers

In [ ]:

async def run_request_batch(
    client: httpx.AsyncClient,
    base_url: str,
    context_length: int,
    concurrency: int,
    request_count: int,
    max_output_tokens: int,
    timeout_s: float,
    phase: str,
) -> List[RequestResult]:
    semaphore = asyncio.Semaphore(concurrency)
    results: List[RequestResult] = []
    prompt = build_prompt_exact_tokens(context_length)

    async def worker(request_id: int):
        async with semaphore:
            result = await single_request(
                client=client,
                base_url=base_url,
                prompt=prompt,
                max_output_tokens=max_output_tokens,
                timeout_s=timeout_s,
                context_length=context_length,
                concurrency=concurrency,
                request_id=request_id,
                phase=phase,
            )
            results.append(result)

    tasks = [asyncio.create_task(worker(i + 1)) for i in range(request_count)]
    await asyncio.gather(*tasks)
    return results

def summarize_results(results: List[RequestResult], wall_clock_s: float) -> Dict[str, Any]:
    latencies = [r.total_latency_ms for r in results if r.success and r.total_latency_ms is not None]
    prompt_tokens = [r.prompt_tokens for r in results if r.success and r.prompt_tokens is not None]
    completion_tokens = [r.completion_tokens for r in results if r.success and r.completion_tokens is not None]
    total_tokens = [r.total_tokens for r in results if r.success and r.total_tokens is not None]
    output_tokens_est = [r.output_tokens_est for r in results if r.success and r.output_tokens_est is not None]

    total = len(results)
    success = sum(1 for r in results if r.success)
    failed = total - success

    total_completion_tokens = int(sum(completion_tokens)) if completion_tokens else None
    total_all_tokens = int(sum(total_tokens)) if total_tokens else None
    total_output_tokens_est = int(sum(output_tokens_est)) if output_tokens_est else None

    return {
        "wall_clock_s": round(wall_clock_s, 4),
        "total_requests": total,
        "success_count": success,
        "failure_count": failed,
        "success_rate_pct": round((success / total) * 100, 2) if total else 0.0,
        "requests_per_sec": round(success / wall_clock_s, 4) if wall_clock_s > 0 else None,
        "latency_ms_avg": round(statistics.mean(latencies), 2) if latencies else None,
        "latency_ms_min": round(min(latencies), 2) if latencies else None,
        "latency_ms_max": round(max(latencies), 2) if latencies else None,
        "latency_ms_p50": round(percentile(latencies, 50), 2) if latencies else None,
        "latency_ms_p95": round(percentile(latencies, 95), 2) if latencies else None,
        "latency_ms_p99": round(percentile(latencies, 99), 2) if latencies else None,
        "avg_prompt_tokens": round(statistics.mean(prompt_tokens), 2) if prompt_tokens else None,
        "avg_completion_tokens": round(statistics.mean(completion_tokens), 2) if completion_tokens else None,
        "avg_total_tokens": round(statistics.mean(total_tokens), 2) if total_tokens else None,
        "completion_tokens_per_sec": round(total_completion_tokens / wall_clock_s, 4) if total_completion_tokens is not None and wall_clock_s > 0 else None,
        "total_tokens_per_sec": round(total_all_tokens / wall_clock_s, 4) if total_all_tokens is not None and wall_clock_s > 0 else None,
        "output_tokens_est_per_sec": round(total_output_tokens_est / wall_clock_s, 4) if total_output_tokens_est is not None and wall_clock_s > 0 else None,
        "sample_errors": [r.error for r in results if not r.success][:5],
    }

def build_summary_dataframe(all_results: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for item in all_results:
        rows.append({
            "context_length": item["context_length"],
            "concurrency": item["concurrency"],
            **item["summary"],
        })
    return pd.DataFrame(rows)

def build_request_dataframe(all_results: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for item in all_results:
        for req in item["requests"]:
            rows.append(dict(req))
    return pd.DataFrame(rows)


## 8. Optional: run a single matrix point first

Useful for trying one specific context + concurrency combination before the full sweep.


In [ ]:

SINGLE_TEST_CONTEXT = CONTEXT_LENGTHS[0]
SINGLE_TEST_CONCURRENCY = CONCURRENCY_LEVELS[0]
SINGLE_TEST_REQUESTS = 5

async def run_single_matrix_demo():
    transport = httpx.AsyncHTTPTransport(retries=0, verify=VERIFY_SSL)
    async with httpx.AsyncClient(transport=transport) as client:
        _ = await run_request_batch(
            client=client,
            base_url=BASE_URL,
            context_length=SINGLE_TEST_CONTEXT,
            concurrency=SINGLE_TEST_CONCURRENCY,
            request_count=1,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            timeout_s=TIMEOUT_SECONDS,
            phase="warmup",
        )

        start = time.perf_counter()
        measured_results = await run_request_batch(
            client=client,
            base_url=BASE_URL,
            context_length=SINGLE_TEST_CONTEXT,
            concurrency=SINGLE_TEST_CONCURRENCY,
            request_count=SINGLE_TEST_REQUESTS,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            timeout_s=TIMEOUT_SECONDS,
            phase="measured",
        )
        wall_clock_s = time.perf_counter() - start
        summary = summarize_results(measured_results, wall_clock_s)
        return measured_results, summary

single_matrix_results, single_matrix_summary = asyncio.run(run_single_matrix_demo())
display(pd.DataFrame([single_matrix_summary]))
display(pd.DataFrame([asdict(x) for x in single_matrix_results]))


## 9. Full benchmark runner

This runs the full context × concurrency matrix with warmup and measured requests.


In [ ]:

async def run_full_benchmark() -> List[Dict[str, Any]]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    all_results = []

    limits = httpx.Limits(max_connections=1000, max_keepalive_connections=100)
    transport = httpx.AsyncHTTPTransport(retries=0, verify=VERIFY_SSL)

    async with httpx.AsyncClient(
        headers={"Content-Type": "application/json"},
        limits=limits,
        transport=transport,
    ) as client:
        for context_length in CONTEXT_LENGTHS:
            for concurrency in CONCURRENCY_LEVELS:
                print(f"[INFO] Warmup: context={context_length}, concurrency={concurrency}")
                warmup_results = await run_request_batch(
                    client=client,
                    base_url=BASE_URL,
                    context_length=context_length,
                    concurrency=concurrency,
                    request_count=WARMUP_REQUESTS,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                    timeout_s=TIMEOUT_SECONDS,
                    phase="warmup",
                )

                print(f"[INFO] Measure: context={context_length}, concurrency={concurrency}")
                start = time.perf_counter()
                measured_results = await run_request_batch(
                    client=client,
                    base_url=BASE_URL,
                    context_length=context_length,
                    concurrency=concurrency,
                    request_count=REQUESTS_PER_TEST,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                    timeout_s=TIMEOUT_SECONDS,
                    phase="measured",
                )
                wall_clock_s = time.perf_counter() - start

                summary = summarize_results(measured_results, wall_clock_s)
                all_results.append({
                    "context_length": context_length,
                    "concurrency": concurrency,
                    "summary": summary,
                    "warmup_requests": [asdict(r) for r in warmup_results],
                    "requests": [asdict(r) for r in measured_results],
                })

                display(pd.DataFrame([{
                    "context_length": context_length,
                    "concurrency": concurrency,
                    **summary
                }]))

    return all_results

# Run this cell when ready
all_results = asyncio.run(run_full_benchmark())
print(f"Completed {len(all_results)} test points.")


## 10. View summary tables in notebook

In [ ]:

summary_df = build_summary_dataframe(all_results)
requests_df = build_request_dataframe(all_results)

display(summary_df)
display(requests_df.head(20))


## 11. Inline plots

These render directly in the notebook and are also saved to PNG files.


In [ ]:

avg_latency_plot = OUTPUT_DIR / "avg_latency_vs_context.png"
p95_latency_plot = OUTPUT_DIR / "p95_latency_vs_context.png"
rps_plot = OUTPUT_DIR / "throughput_vs_concurrency.png"
tokens_per_sec_plot = OUTPUT_DIR / "completion_tokens_per_sec_vs_concurrency.png"

# Average latency vs context
plt.figure(figsize=(10, 6))
for concurrency in sorted(summary_df["concurrency"].unique()):
    subset = summary_df[summary_df["concurrency"] == concurrency].sort_values("context_length")
    plt.plot(subset["context_length"], subset["latency_ms_avg"], marker="o", label=f"Concurrency {concurrency}")
plt.xlabel("Context Length")
plt.ylabel("Average Latency (ms)")
plt.title("Average Latency vs Context Length")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(avg_latency_plot, dpi=150, bbox_inches="tight")
plt.show()

# P95 latency vs context
plt.figure(figsize=(10, 6))
for concurrency in sorted(summary_df["concurrency"].unique()):
    subset = summary_df[summary_df["concurrency"] == concurrency].sort_values("context_length")
    plt.plot(subset["context_length"], subset["latency_ms_p95"], marker="o", label=f"Concurrency {concurrency}")
plt.xlabel("Context Length")
plt.ylabel("P95 Latency (ms)")
plt.title("P95 Latency vs Context Length")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(p95_latency_plot, dpi=150, bbox_inches="tight")
plt.show()

# Throughput vs concurrency
plt.figure(figsize=(10, 6))
for context_length in sorted(summary_df["context_length"].unique()):
    subset = summary_df[summary_df["context_length"] == context_length].sort_values("concurrency")
    plt.plot(subset["concurrency"], subset["requests_per_sec"], marker="o", label=f"Context {context_length}")
plt.xlabel("Concurrency")
plt.ylabel("Successful Requests / sec")
plt.title("Throughput vs Concurrency")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(rps_plot, dpi=150, bbox_inches="tight")
plt.show()

# Completion tokens per sec
plt.figure(figsize=(10, 6))
for context_length in sorted(summary_df["context_length"].unique()):
    subset = summary_df[summary_df["context_length"] == context_length].sort_values("concurrency")
    plt.plot(subset["concurrency"], subset["completion_tokens_per_sec"], marker="o", label=f"Context {context_length}")
plt.xlabel("Concurrency")
plt.ylabel("Completion Tokens / sec")
plt.title("Completion Token Throughput vs Concurrency")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(tokens_per_sec_plot, dpi=150, bbox_inches="tight")
plt.show()

print("Saved plots to:")
print(avg_latency_plot)
print(p95_latency_plot)
print(rps_plot)
print(tokens_per_sec_plot)


## 12. Save JSON and CSV outputs

In [ ]:

detailed_json = OUTPUT_DIR / "benchmark_results.json"
summary_csv = OUTPUT_DIR / "benchmark_summary.csv"
requests_csv = OUTPUT_DIR / "benchmark_requests.csv"

with open(detailed_json, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

summary_df.to_csv(summary_csv, index=False)
requests_df.to_csv(requests_csv, index=False)

print("Saved:")
print(detailed_json)
print(summary_csv)
print(requests_csv)


## 13. Build and preview HTML report in notebook

This creates a client-shareable HTML report and also renders a preview directly below the cell.


In [ ]:

def make_html_table(df: pd.DataFrame) -> str:
    display_cols = [
        "context_length", "concurrency", "success_rate_pct", "requests_per_sec",
        "latency_ms_avg", "latency_ms_p95", "latency_ms_p99",
        "avg_prompt_tokens", "avg_completion_tokens", "completion_tokens_per_sec"
    ]
    cols = [c for c in display_cols if c in df.columns]
    return df[cols].round(2).to_html(index=False, border=0, classes="summary-table")

def build_findings(df: pd.DataFrame) -> str:
    if df.empty:
        return "<li>No results available.</li>"
    best_p95_row = df.dropna(subset=["latency_ms_p95"]).sort_values("latency_ms_p95").iloc[0]
    best_rps_row = df.dropna(subset=["requests_per_sec"]).sort_values("requests_per_sec", ascending=False).iloc[0]
    most_stable = df.sort_values(["failure_count", "latency_ms_p95"]).iloc[0]
    findings = [
        f"Lowest observed P95 latency was at context {int(best_p95_row['context_length'])} and concurrency {int(best_p95_row['concurrency'])}, with P95 latency of {best_p95_row['latency_ms_p95']:.2f} ms.",
        f"Highest successful request throughput was at context {int(best_rps_row['context_length'])} and concurrency {int(best_rps_row['concurrency'])}, reaching {best_rps_row['requests_per_sec']:.2f} req/s.",
        f"Most stable combination by failures then P95 latency was context {int(most_stable['context_length'])} and concurrency {int(most_stable['concurrency'])}.",
    ]
    return "".join(f"<li>{x}</li>" for x in findings)

report_html_path = OUTPUT_DIR / "benchmark_report.html"

html = f'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8" />
<title>LLM Benchmark Report</title>
<style>
body {{
  font-family: Arial, sans-serif;
  margin: 32px;
  color: #222;
}}
h1, h2 {{
  margin-bottom: 8px;
}}
p, li {{
  line-height: 1.5;
}}
.section {{
  margin-top: 28px;
}}
.kpi-grid {{
  display: grid;
  grid-template-columns: repeat(4, minmax(180px, 1fr));
  gap: 12px;
}}
.kpi {{
  border: 1px solid #ddd;
  border-radius: 8px;
  padding: 14px;
  background: #fafafa;
}}
.kpi .label {{
  font-size: 12px;
  color: #666;
}}
.kpi .value {{
  font-size: 24px;
  font-weight: bold;
  margin-top: 6px;
}}
.summary-table {{
  border-collapse: collapse;
  width: 100%;
  margin-top: 12px;
}}
.summary-table th, .summary-table td {{
  border: 1px solid #ddd;
  padding: 8px;
  text-align: right;
}}
.summary-table th:first-child, .summary-table td:first-child {{
  text-align: left;
}}
.summary-table th:nth-child(2), .summary-table td:nth-child(2) {{
  text-align: left;
}}
img {{
  max-width: 100%;
  border: 1px solid #ddd;
  border-radius: 6px;
  margin-top: 10px;
}}
code {{
  background: #f5f5f5;
  padding: 2px 4px;
  border-radius: 4px;
}}
</style>
</head>
<body>
  <h1>LLM Benchmark Report</h1>
  <p><strong>Generated:</strong> {datetime.now().isoformat(timespec="seconds")}</p>

  <div class="section">
    <h2>Test Configuration</h2>
    <ul>
      <li><strong>Base URL:</strong> <code>{BASE_URL}</code></li>
      <li><strong>Model:</strong> <code>{MODEL_NAME}</code></li>
      <li><strong>Endpoint:</strong> <code>/v1/completions</code></li>
      <li><strong>Context lengths:</strong> {CONTEXT_LENGTHS}</li>
      <li><strong>Concurrency levels:</strong> {CONCURRENCY_LEVELS}</li>
      <li><strong>Measured requests per test:</strong> {REQUESTS_PER_TEST}</li>
      <li><strong>Warmup requests per test:</strong> {WARMUP_REQUESTS}</li>
      <li><strong>Max output tokens:</strong> {MAX_OUTPUT_TOKENS}</li>
      <li><strong>SSL verification:</strong> {VERIFY_SSL}</li>
    </ul>
  </div>

  <div class="section">
    <h2>Executive Summary</h2>
    <div class="kpi-grid">
      <div class="kpi"><div class="label">Total Test Points</div><div class="value">{len(summary_df)}</div></div>
      <div class="kpi"><div class="label">Best P95 Latency (ms)</div><div class="value">{summary_df['latency_ms_p95'].min():.2f}</div></div>
      <div class="kpi"><div class="label">Best Throughput (req/s)</div><div class="value">{summary_df['requests_per_sec'].max():.2f}</div></div>
      <div class="kpi"><div class="label">Worst Failure Count</div><div class="value">{int(summary_df['failure_count'].max())}</div></div>
    </div>
    <ul>
      {build_findings(summary_df)}
    </ul>
  </div>

  <div class="section">
    <h2>Summary Table</h2>
    {make_html_table(summary_df)}
  </div>

  <div class="section">
    <h2>Charts</h2>
    <h3>Average Latency vs Context Length</h3>
    <img src="{avg_latency_plot.name}" alt="Average latency plot" />
    <h3>P95 Latency vs Context Length</h3>
    <img src="{p95_latency_plot.name}" alt="P95 latency plot" />
    <h3>Throughput vs Concurrency</h3>
    <img src="{rps_plot.name}" alt="Requests per second plot" />
    <h3>Completion Token Throughput vs Concurrency</h3>
    <img src="{tokens_per_sec_plot.name}" alt="Tokens per second plot" />
  </div>

  <div class="section">
    <h2>Generated Files</h2>
    <ul>
      <li>Detailed JSON: <code>{detailed_json.name}</code></li>
      <li>Summary CSV: <code>{summary_csv.name}</code></li>
      <li>Per-request CSV: <code>{requests_csv.name}</code></li>
    </ul>
  </div>
</body>
</html>
'''

report_html_path.write_text(html, encoding="utf-8")
print("Saved HTML report to:", report_html_path)

display(HTML(html))


## 14. Final file locations

In [ ]:

files_created = [
    OUTPUT_DIR / "benchmark_results.json",
    OUTPUT_DIR / "benchmark_summary.csv",
    OUTPUT_DIR / "benchmark_requests.csv",
    OUTPUT_DIR / "avg_latency_vs_context.png",
    OUTPUT_DIR / "p95_latency_vs_context.png",
    OUTPUT_DIR / "throughput_vs_concurrency.png",
    OUTPUT_DIR / "completion_tokens_per_sec_vs_concurrency.png",
    OUTPUT_DIR / "benchmark_report.html",
]

for path in files_created:
    print(path.resolve())
